# MAST download notebook

Query MAST, inspect sky coverage, and download HST or JWST science products.
Shared logic lives in `st123.stages.download` and `st123.utils.helpers` / `st123.utils`.

CLI equivalent: `python -m st123.scripts.download --ra ... --dec ... --obj ...`


In [ ]:
import sys
import os
from pathlib import Path

# Resolve repo root whether cwd is repo, st123/, or st123/notebooks/
_here = Path.cwd().resolve()
ROOT = None
for candidate in [_here, *_here.parents]:
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'st123').is_dir():
        ROOT = candidate
        break
if ROOT is None:
    ROOT = _here
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import shapely
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.mast import Observations

from st123.stages.download import (
    collect_hst_products,
    coverage_fraction,
    download_jwst_observations,
    filter_jwst_products,
    polygons_from_obs_table,
    query_hst,
    query_jwst,
    resolve_mast_token,
)
from st123.utils.helpers import input_list, organize_reduction_tables, pick_deepest_images


## Target and options

Set `DOWNLOAD = True` only when you want products written to disk.


In [ ]:
TARGETS = {
    'NGC4536': (189.9976, -11.623),
    'M92': (259.2800254, 43.13566),
    'NGC628': (24.174049, 15.78346),
    'NGC5457': (210.803, 54.34906),
}

obj = 'NGC4536'
ra, dec = TARGETS[obj]
coord = SkyCoord(ra, dec, frame='icrs', unit=u.deg)
radius = 12 * u.arcmin

DOWNLOAD = False
# MAST token for proprietary data (or set MAST_API_TOKEN / MAST_TOKEN).
# Create one at https://auth.mast.stsci.edu/info
TOKEN = None  # e.g. 'your-mast-token'
TOKEN = resolve_mast_token(TOKEN)

HST_OUTDIR = Path(f'hst_data/{obj}')
JWST_OUTDIR = Path(f'jwst_data/{obj}')
JWST_STAGE = 2  # 2=CAL, 3=I2D

coord, radius, obj, TOKEN


## HST query and download


In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

hst_table = query_hst(
    coord,
    radius=radius,
    filters=['F275W', 'F555W', 'F814W'],
    token=TOKEN,
)
logger.info(len(hst_table), 'HST imaging observations')
hst_table[:5]


In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

productlist = collect_hst_products(hst_table[:6])
logger.info(0 if productlist is None else len(productlist), 'science products in preview subset')
productlist

In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

if DOWNLOAD and productlist is not None and len(productlist):
    HST_OUTDIR.mkdir(parents=True, exist_ok=True)
    Observations.download_products(productlist, download_dir=str(HST_OUTDIR), extension='fits')
    logger.info('Downloaded HST products to', HST_OUTDIR)
else:
    logger.info('Skipping HST download (set DOWNLOAD=True to enable).')

## JWST query, coverage, and download


In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

jwst_table = query_jwst(coord, radius=radius, token=TOKEN)
logger.info(len(jwst_table), 'JWST imaging observations')
jwst_table[:5]


In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

pgons, filters = polygons_from_obs_table(jwst_table)
net_field = shapely.unary_union(pgons)

want = ['F090W', 'F115W', 'F150W', 'F200W']
mask = [str(f) in want for f in filters]
logger.info('coverage fraction for', want, ':', coverage_fraction(pgons, mask))
net_field

In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

plist = filter_jwst_products(
    Observations.get_product_list(jwst_table[0]),
    stage=JWST_STAGE,
)
logger.info(len(plist), 'products for first observation at stage', JWST_STAGE)
plist

In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

if DOWNLOAD and len(jwst_table):
    download_jwst_observations(
        jwst_table,
        outdir=str(JWST_OUTDIR),
        stage=JWST_STAGE,
        token=TOKEN,
    )
    logger.info('Downloaded JWST products to', JWST_OUTDIR)
else:
    logger.info('Skipping JWST download (set DOWNLOAD=True to enable).')


## Local image bookkeeping

After FITS files are on disk, build an observation table and pick deep frames.


In [ ]:
import logging
logger = logging.getLogger('st123.notebook')
if not logging.getLogger("st123").handlers:
    logging.basicConfig(level=logging.INFO)

search_dirs = [JWST_OUTDIR, HST_OUTDIR, Path('.')]
images = []
for d in search_dirs:
    if d.is_dir():
        images.extend(sorted(str(p) for p in d.rglob('*_cal.fits')))
        images.extend(sorted(str(p) for p in d.rglob('*_flc.fits')))
        images.extend(sorted(str(p) for p in d.rglob('*_flt.fits')))
images = sorted(set(images))

if not images:
    logger.info('No local CAL/FLC/FLT files found yet; download first or point search_dirs at your data.')
    obstable = None
else:
    obstable = input_list(images)
    tables = organize_reduction_tables(obstable, byvisit=True, bymodule=True)
    deepest = pick_deepest_images(images)
    logger.info(len(images), 'images;', len(tables), 'reduction groups')
    logger.info('deepest:', deepest)

obstable
